# 01 Interpolation Overview

Interpolation starts from discrete data points

$$
(x_0,y_0), (x_1,y_1), \dots, (x_n,y_n),
$$

and constructs a function $p(x)$ that satisfies

$$
p(x_i)=y_i,\qquad i=0,1,\dots,n.
$$

This notebook sets up the problem, contrasts interpolation with fitting, and
shows why node placement matters before any algorithm is chosen.


## Interpolation, fitting, and data trust

Interpolation is exact at the nodes. This is appropriate for table values,
exact function samples, or benchmark data. Fitting is different: it allows
residuals and is often better when measurements contain noise.

The numerical task is therefore not just "draw a curve." We must decide:

* what data are trusted;
* what function space is used;
* how the node distribution affects stability;
* whether local or global behavior is more important.


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import lagrange_interpolate, piecewise_linear_interpolate

x = np.array([0.0, 0.7, 1.5, 2.4, 3.0, 4.0])
y = np.sin(x) + 0.15 * x
x_eval = np.linspace(x.min(), x.max(), 400)

poly = lagrange_interpolate(x, y, x_eval)
linear = piecewise_linear_interpolate(x, y, x_eval)

plt.figure(figsize=(8, 4.5))
plt.plot(x_eval, poly, label="Global polynomial")
plt.plot(x_eval, linear, label="Piecewise linear")
plt.scatter(x, y, color="black", zorder=3, label="Data")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Two interpolants for the same data")
plt.legend()
plt.tight_layout()
plt.show()


## Interpolation is not the same as least-squares fitting

If data are noisy, forcing a curve through every point can make the curve follow
measurement noise. A low-degree least-squares fit can miss local details, but it
may better represent the broad trend.


In [ ]:
rng = np.random.default_rng(2026)
x_noisy = np.linspace(0.0, 4.0, 9)
y_clean = np.sin(x_noisy) + 0.15 * x_noisy
y_noisy = y_clean + 0.12 * rng.normal(size=x_noisy.size)

x_plot = np.linspace(x_noisy.min(), x_noisy.max(), 500)
interpolant = lagrange_interpolate(x_noisy, y_noisy, x_plot)
fit_coefficients = np.polyfit(x_noisy, y_noisy, deg=3)
least_squares_fit = np.polyval(fit_coefficients, x_plot)

plt.figure(figsize=(8, 4.5))
plt.plot(x_plot, interpolant, label="Degree-8 interpolant")
plt.plot(x_plot, least_squares_fit, label="Degree-3 least-squares fit")
plt.scatter(x_noisy, y_noisy, color="black", zorder=3, label="Noisy data")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Exact interpolation can amplify noisy samples")
plt.legend()
plt.tight_layout()
plt.show()


## Interpolation space and uniqueness

For $n+1$ distinct nodes, there is exactly one polynomial in $\mathcal P_n$ that
interpolates the data. One way to see this computationally is through the
Vandermonde matrix. If the nodes are distinct, the matrix is invertible, but its
condition number can still be large.


In [ ]:
def vandermonde_condition(nodes):
    matrix = np.vander(nodes, N=nodes.size, increasing=True)
    return np.linalg.cond(matrix)

for n in [5, 9, 13]:
    equal_nodes = np.linspace(-1.0, 1.0, n)
    print(f"n={n:2d}, cond(V)={vandermonde_condition(equal_nodes):10.3e}")


## Summary

* Interpolation enforces the data values exactly.
* Fitting is often better for noisy data.
* Polynomial interpolation is unique for distinct nodes, but uniqueness does not
  guarantee numerical stability.
* The next notebook studies global polynomial interpolation in detail.
